# GeoChange AI — STEP 2B/2C: Pipeline visual check

This notebook:
- Loads the encoded bytes using `src/data.py`.
- Prints one train, validation and test batch description.
- Plots four augmented training examples and four unaugmented validation examples.
- Times 50 training batches.
- Does not define or train a model.

The plotted RGB images are converted back from MobileNetV2's `[-1, 1]` range for display only.


In [ ]:
# 1. Mount Drive and point to the project

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_DIR = Path("/content/drive/MyDrive/GeoChange-AI")
sys.path.insert(0, str(PROJECT_DIR))

from src.data import (
    load_data,
    make_dataset,
    describe_batch,
    time_batches,
)

print("Project:", PROJECT_DIR)


In [ ]:
# 2. Load encoded bytes and create the three datasets

load_summary = load_data(PROJECT_DIR / "configs" / "default.yaml")
print("Loaded sample counts:", load_summary["counts"])

train_ds = make_dataset(
    "train",
    batch_size=16,
    training=True,
    seed=42,
)

val_ds = make_dataset(
    "val",
    batch_size=16,
    training=False,
    seed=42,
)

test_ds = make_dataset(
    "test",
    batch_size=16,
    training=False,
    seed=42,
)


In [ ]:
# 3. Describe one batch from each split

print("\nTRAIN BATCH")
describe_batch(train_ds)

print("\nVALIDATION BATCH")
describe_batch(val_ds)

print("\nTEST BATCH")
describe_batch(test_ds)


In [ ]:
# 4. Plot four augmented training samples: imageA | imageB | mask

(train_a, train_b), train_masks = next(iter(train_ds))

def display_mobilenet_image(image_tensor):
    # MobileNetV2 preprocessing maps RGB [0,255] to [-1,1].
    # Reverse that mapping only for human-readable plotting.
    return tf.clip_by_value((image_tensor + 1.0) / 2.0, 0.0, 1.0).numpy()

fig, axes = plt.subplots(4, 3, figsize=(10, 13))

for i in range(4):
    axes[i, 0].imshow(display_mobilenet_image(train_a[i]))
    axes[i, 0].set_title(f"Train {i+1} — imageA")

    axes[i, 1].imshow(display_mobilenet_image(train_b[i]))
    axes[i, 1].set_title(f"Train {i+1} — imageB")

    axes[i, 2].imshow(train_masks[i, ..., 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title(f"Train {i+1} — augmented mask")

    for j in range(3):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 5. Plot four unaugmented validation samples: imageA | imageB | mask

(val_a, val_b), val_masks = next(iter(val_ds))

fig, axes = plt.subplots(4, 3, figsize=(10, 13))

for i in range(4):
    axes[i, 0].imshow(display_mobilenet_image(val_a[i]))
    axes[i, 0].set_title(f"Validation {i+1} — imageA")

    axes[i, 1].imshow(display_mobilenet_image(val_b[i]))
    axes[i, 1].set_title(f"Validation {i+1} — imageB")

    axes[i, 2].imshow(val_masks[i, ..., 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title(f"Validation {i+1} — original mask")

    for j in range(3):
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 6. Time 50 training batches

timing_result = time_batches(train_ds, batches=50)


## Pytest

Run this in a separate Colab cell after saving `src/data.py` and `tests/test_preprocess.py`:

```python
%cd /content/drive/MyDrive/GeoChange-AI
!pip -q install pytest pyyaml pyarrow
!python -m pytest -q tests/test_preprocess.py
```

Expected test count: 9. The actual pass/fail output depends on the runtime and must be checked after execution.


In [ ]:
# 7. Short paste-back summary

print("\n========== GEOCHANGE AI — STEP 2B/2C SUMMARY ==========")
print("Loaded counts:", load_summary["counts"])
print("Batch size:", 16)
print("Seed:", 42)
print("Image size:", 256)
print("Training augmentation: shared flip-left/right, flip-up/down and rot90")
print("Validation/test: deterministic, no augmentation, no shuffle")
print("Mask rule: any nonzero PNG value becomes 1.0")
print("MobileNetV2 preprocessing: imageA/imageB only")
print("Decoded cache: not used")
print("Timing result:", timing_result)
print("Pytest: run the separate pytest command above and paste its output")
print("========================================================")
